# Old -> New Annotation Conversion Rule

**The generalized rule** (confirmed correct — see validation below and manual review of samples 8, 9, 10):

For every question in a document, in one pass:

1. If `question.text` is already non-empty, leave it unchanged.
2. If `question.text` is empty:
   - If its own section has a non-empty `title`, set `question.text = section.title`.
   - Otherwise (section title also empty), set `question.text = document.title` (only the first time this happens), and clear `document.title = ""`.

No section-collapsing, no per-sample logic, no sample numbers anywhere in the rule — it only ever looks at whether a given piece of text is empty and what the nearest non-empty title is.

This notebook is exploratory only — nothing here is wired into the package/pipeline.

In [1]:
import copy
import json
from pathlib import Path

from dmpbridge.evaluation.evaluate import LLM_DIR, MANUAL_DIR, NEW_MANUAL_DIR

## The conversion function

In [2]:
def apply_new_annotation_rules(data: dict) -> dict:
    """Return a copy of *data* with Rules 1 and 3 applied (see notebook intro)."""
    data     = copy.deepcopy(data)
    root     = data.get('narrative', data)
    template = root.get('template', {})
    doc_title  = template.get('title', '').strip()
    title_used = False

    for section in template.get('section', []):
        sec_title = section.get('title', '').strip()
        for question in section.get('question', []):
            if question.get('text', '').strip():
                continue
            if sec_title:
                question['text'] = sec_title
            elif doc_title and not title_used:
                question['text'] = doc_title
                title_used = True

    if title_used:
        template['title'] = ''

    return data

## Validate against real ground truth

For the 6 samples where no section-collapse happened (1, 2, 3, 4, 7, and their counterparts), the converted old-version file should match the actual new-version file **exactly**.

In [3]:
NON_COLLAPSE_SAMPLES = [1, 2, 3, 4, 7]

for n in NON_COLLAPSE_SAMPLES:
    old = json.loads((MANUAL_DIR / f'sample{n}_old_dmp.json').read_text(encoding='utf-8'))
    new = json.loads((NEW_MANUAL_DIR / f'sample{n}_new_dmp.json').read_text(encoding='utf-8'))
    got = apply_new_annotation_rules(old)
    print(f'sample{n}: exact match with real new-version ground truth? {got == new}')

sample1: exact match with real new-version ground truth? True
sample2: exact match with real new-version ground truth? True
sample3: exact match with real new-version ground truth? True
sample4: exact match with real new-version ground truth? True
sample7: exact match with real new-version ground truth? True


For samples 5, 6, 8, 9, 10, the original new-annotation ground truth happened to also collapse multiple sections into one — but manual review of the converted output (e.g. sample 8) confirmed that's not required: keeping each section separate with its own backfilled question is an acceptable, correct annotation on its own. So these won't exact-match the original new-version file section-count-wise, and that's expected and fine.

In [4]:
COLLAPSE_SAMPLES = [5, 6, 8, 9, 10]

for n in COLLAPSE_SAMPLES:
    old = json.loads((MANUAL_DIR / f'sample{n}_old_dmp.json').read_text(encoding='utf-8'))
    new = json.loads((NEW_MANUAL_DIR / f'sample{n}_new_dmp.json').read_text(encoding='utf-8'))
    got = apply_new_annotation_rules(old)
    t_got, t_new = got['narrative']['template'], new['narrative']['template']
    print(f"sample{n}: sections got={len(t_got['section'])} vs new={len(t_new['section'])}  "
          f"(exact match: {got == new}, expected False)")

sample5: sections got=6 vs new=1  (exact match: False, expected False)
sample6: sections got=5 vs new=1  (exact match: False, expected False)
sample8: sections got=6 vs new=1  (exact match: False, expected False)
sample9: sections got=5 vs new=1  (exact match: False, expected False)
sample10: sections got=6 vs new=1  (exact match: False, expected False)


## Convert & save: old ground truth -> new format (all 10 samples)

Applies the rules to every `sampleN_old_dmp.json` and saves the result to `data/output/ground_truth_converted_test/sampleN_dmp.json`, so you can open any of them directly. Also reports, per sample, whether it matches the real new-version annotation exactly (expected `True` for 1, 2, 3, 4, 7 and `False` for the 5 collapse-case samples, per the validation above).

In [5]:
GT_OUTPUT_DIR = LLM_DIR.parent / 'ground_truth_converted_test'
GT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for n in range(1, 11):
    old = json.loads((MANUAL_DIR / f'sample{n}_old_dmp.json').read_text(encoding='utf-8'))
    new = json.loads((NEW_MANUAL_DIR / f'sample{n}_new_dmp.json').read_text(encoding='utf-8'))
    converted = apply_new_annotation_rules(old)

    out_path = GT_OUTPUT_DIR / f'sample{n}_dmp.json'
    out_path.write_text(json.dumps(converted, indent=2, ensure_ascii=False), encoding='utf-8')

    exact          = converted == new
    n_sections_got = len(converted['narrative']['template']['section'])
    n_sections_new = len(new['narrative']['template']['section'])
    print(f'sample{n:<2}  exact match: {str(exact):5s}  '
          f'sections (converted={n_sections_got}, real-new={n_sections_new})  ->  {out_path.name}')

print(f'\nSaved 10 files under: {GT_OUTPUT_DIR}')

sample1   exact match: True   sections (converted=6, real-new=6)  ->  sample1_dmp.json
sample2   exact match: True   sections (converted=4, real-new=4)  ->  sample2_dmp.json
sample3   exact match: True   sections (converted=5, real-new=5)  ->  sample3_dmp.json
sample4   exact match: True   sections (converted=1, real-new=1)  ->  sample4_dmp.json
sample5   exact match: False  sections (converted=6, real-new=1)  ->  sample5_dmp.json
sample6   exact match: False  sections (converted=5, real-new=1)  ->  sample6_dmp.json
sample7   exact match: True   sections (converted=1, real-new=1)  ->  sample7_dmp.json
sample8   exact match: False  sections (converted=6, real-new=1)  ->  sample8_dmp.json
sample9   exact match: False  sections (converted=5, real-new=1)  ->  sample9_dmp.json
sample10  exact match: False  sections (converted=6, real-new=1)  ->  sample10_dmp.json

Saved 10 files under: C:\Users\Nahid\dmpbridge\data\output\ground_truth_converted_test
